In [4]:
!pip install boto3 anthropic pypdf -q

In [7]:
import boto3
from google.colab import userdata
from anthropic import AnthropicBedrock

aws_key = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret = userdata.get('AWS_SECRET_ACCESS_KEY')
aws_region = userdata.get('AWS_REGION')

client = AnthropicBedrock(
    aws_access_key=aws_key,
    aws_secret_key=aws_secret,
    aws_region=aws_region,
)

response = client.messages.create(
    model="us.anthropic.claude-sonnet-4-6",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Reply with exactly: 'Bedrock connection works. Ready to build.'"}
    ]
)

print(response.content[0].text)

Bedrock connection works. Ready to build.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/csrd-analyzer'
os.makedirs(f'{PROJECT_ROOT}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/extracted', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/text', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/outputs', exist_ok=True)

print(f"Project folder ready at: {PROJECT_ROOT}")
print("Folders inside:", os.listdir(PROJECT_ROOT))

Project folder ready at: /content/drive/MyDrive/csrd-analyzer
Folders inside: ['data', 'outputs']


In [10]:
import os
raw_dir = f'{PROJECT_ROOT}/data/raw'
files = os.listdir(raw_dir)
print(f"Files in {raw_dir}:")
for f in files:
    size_mb = os.path.getsize(f'{raw_dir}/{f}') / (1024 * 1024)
    print(f"  {f} — {size_mb:.1f} MB")

Files in /content/drive/MyDrive/csrd-analyzer/data/raw:
  unilever_2024.pdf — 16.2 MB
  schneider_2024.pdf — 14.0 MB


In [11]:
from pypdf import PdfReader

unilever_path = f'{PROJECT_ROOT}/data/raw/unilever_2024.pdf'

reader = PdfReader(unilever_path)
total_pages = len(reader.pages)
print(f"Unilever report has {total_pages} pages")

# Extract all text, keeping track of page numbers
pages_text = []
for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    pages_text.append({"page": i + 1, "text": text})

# Quick sanity check — look at a middle page
print(f"\n--- Sample from page {total_pages // 2} ---")
print(pages_text[total_pages // 2]["text"][:500])

Unilever report has 305 pages

--- Sample from page 152 ---
4B. PENSIONS AND SIMILAR OBLIGATIONS
For defined benefit plans, operating and finance costs are recognised separately in the income statement. The amount charged to operating 
cost in the income statement is the cost of accruing pension benefits promised to employees over the year, administration costs (other than 
costs of managing plan assets), plus the costs of individual events such as past service benefit changes, settlements and curtailments (such 
events are recognised immediately in the 


In [12]:
import re

# Keywords that signal ESRS E1 climate disclosure section
e1_signals = [
    "ESRS E1",
    "E1 Climate",
    "Climate change",
    "GHG emissions",
    "Scope 1",
    "Scope 2",
    "Scope 3",
    "transition plan",
    "carbon footprint",
]

# Score each page by how many signals it contains
scored_pages = []
for p in pages_text:
    text_lower = p["text"].lower()
    score = sum(1 for sig in e1_signals if sig.lower() in text_lower)
    scored_pages.append({"page": p["page"], "score": score, "text": p["text"]})

# Find pages with high signal density
high_signal = [p for p in scored_pages if p["score"] >= 3]
print(f"Found {len(high_signal)} pages with strong climate signal (score >= 3)")
print(f"\nTop 10 pages by score:")
top_10 = sorted(scored_pages, key=lambda x: x["score"], reverse=True)[:10]
for p in top_10:
    preview = p["text"][:100].replace("\n", " ")
    print(f"  Page {p['page']} (score {p['score']}): {preview}...")

Found 14 pages with strong climate signal (score >= 3)

Top 10 pages by score:
  Page 247 (score 6): Gross Scope 1, 2 and 3, and total GHG emissions  Total GHG emissions are calculated using the GHG Pr...
  Page 298 (score 5): DISCLOSURE REQUIREMENTS COVERED BY OUR SUSTAINABILITY STATEMENT, INCLUDING  INCORPORATION BY REFEREN...
  Page 39 (score 4): MORE FOCUSED, URGENT AND SYSTEMIC Sustainability is a strategic imperative for our business and  a k...
  Page 243 (score 4): IMPACT, RISK AND OPPORTUNITY MANAGEMENT Policies Unilever’s climate policies, which include policies...
  Page 246 (score 4): Scope 1 and 2 target performance  The percentage change in Scope 1 and 2 market-based GHG emissions ...
  Page 248 (score 4):   Category 11 – Use of sold products HFC propellant volumes for aerosol products produced by Unileve...
  Page 301 (score 4): EU LEGISLATION DATA POINTS Disclosure  requirement Data point SFDR reference Pillar 3  reference Ben...
  Page 51 (score 3):   Climate Goal 2024

In [13]:
# Get all pages with at least some climate signal, in order
climate_pages = [p for p in scored_pages if p["score"] >= 2]

# Combine into one block of text, with page markers so Claude can cite them
climate_text = "\n\n".join(
    f"[Page {p['page']}]\n{p['text']}" for p in climate_pages
)

# Estimate tokens (rough rule: ~4 chars per token)
estimated_tokens = len(climate_text) // 4
print(f"Climate section: {len(climate_pages)} pages, {len(climate_text):,} characters, ~{estimated_tokens:,} tokens")

# Save to disk so we don't re-extract every time
with open(f'{PROJECT_ROOT}/data/text/unilever_climate_section.txt', 'w') as f:
    f.write(climate_text)

print("Saved to data/text/unilever_climate_section.txt")
print(f"\n--- First 800 chars ---\n{climate_text[:800]}")

Climate section: 21 pages, 114,780 characters, ~28,695 tokens
Saved to data/text/unilever_climate_section.txt

--- First 800 chars ---
[Page 10]
ensure we take better advantage of these positions. Brands 
will be prioritised according to their ability to grow their 
categories and gain share. This will be done by focusing on 
fewer, bigger, more scalable innovations, and through the 
continued rapid roll-out globally of the Unmissable Brand 
Superiority (UBS) framework, which allows us to address 
elements of underperformance quickly and holistically across 
six areas of consumer preference: product, price, packaging, 
proposition, promotion and place. 
The choices we made and implemented in 2024 have put our 
portfolio in good shape for the future. The planned separation 
of Ice Cream, the sale of the Russian business, and the 
disposals of Elida Beauty, Truliva and Pureit mean we can 
focus on our four excellent Busines


In [14]:
import json

# Load the climate text we saved
with open(f'{PROJECT_ROOT}/data/text/unilever_climate_section.txt', 'r') as f:
    climate_text = f.read()

# The extraction schema — what we want Claude to pull out
extraction_prompt = f"""You are a sustainability data analyst extracting structured climate disclosures from a corporate sustainability report. The report is from Unilever, fiscal year 2024.

Below is the climate-relevant section of their report (extracted with [Page N] markers showing original page numbers).

Your task: extract the following data points into clean JSON. For every numeric value, include the source page number. If a data point is not disclosed or you cannot find it with high confidence, set the value to null. Do not guess. Do not infer.

REQUIRED FIELDS:

1. ghg_emissions:
   - scope_1_tco2e (with page)
   - scope_2_location_based_tco2e (with page)
   - scope_2_market_based_tco2e (with page)
   - scope_3_total_tco2e (with page)
   - scope_3_categories: list of objects with category_number (1-15), category_name, value_tco2e, page

2. emissions_targets:
   - has_sbti_validation (true/false/null)
   - sbti_target_description (string or null)
   - net_zero_target_year (int or null)
   - scope_1_2_reduction_target_pct (number or null)
   - scope_1_2_baseline_year (int or null)
   - scope_3_reduction_target_pct (number or null)
   - scope_3_baseline_year (int or null)

3. transition_plan:
   - has_transition_plan (true/false/null)
   - key_levers (list of strings, max 5)
   - capex_aligned_to_transition (string or null — verbatim quote if disclosed)

4. internal_carbon_price:
   - is_used (true/false/null)
   - price_per_tco2e (number or null)
   - currency (string or null)
   - scope_of_application (string or null)

5. methodology:
   - calculation_standard (e.g. "GHG Protocol", string or null)
   - assurance_provider (string or null)
   - assurance_level (e.g. "limited", "reasonable", null)

6. data_quality_flags:
   - List any inconsistencies, ambiguities, or notable concerns you encountered while extracting (e.g. "Scope 3 total disclosed on page 247 but category sum on page 248 differs by X tCO2e"). Be specific. If none, return an empty list.

OUTPUT: Return ONLY valid JSON, no markdown formatting, no preamble, no explanation. The JSON object should have all six top-level keys above.

REPORT TEXT:
\"\"\"
{climate_text}
\"\"\"
"""

print(f"Prompt size: ~{len(extraction_prompt) // 4:,} tokens")
print("Sending to Claude... (this takes 30-60 seconds for a response of this size)\n")

response = client.messages.create(
    model="us.anthropic.claude-sonnet-4-6",
    max_tokens=4000,
    messages=[{"role": "user", "content": extraction_prompt}]
)

raw_output = response.content[0].text
print("--- Raw output from Claude ---")
print(raw_output[:2000])
print(f"\n--- Token usage ---")
print(f"Input tokens: {response.usage.input_tokens:,}")
print(f"Output tokens: {response.usage.output_tokens:,}")

# Rough cost estimate (Sonnet 4.6 on Bedrock: ~$3/M input, ~$15/M output)
cost = (response.usage.input_tokens / 1_000_000 * 3) + (response.usage.output_tokens / 1_000_000 * 15)
print(f"Estimated cost: ${cost:.4f}")

Prompt size: ~29,228 tokens
Sending to Claude... (this takes 30-60 seconds for a response of this size)

--- Raw output from Claude ---
{
  "ghg_emissions": {
    "scope_1_tco2e": {
      "value": 480000,
      "unit": "tCO2e",
      "note": "0.48 million tonnes CO2e",
      "page": 248
    },
    "scope_2_location_based_tco2e": {
      "value": 1260000,
      "unit": "tCO2e",
      "note": "1.26 million tonnes CO2e",
      "page": 248
    },
    "scope_2_market_based_tco2e": {
      "value": 210000,
      "unit": "tCO2e",
      "note": "0.21 million tonnes CO2e",
      "page": 248
    },
    "scope_3_total_tco2e": {
      "value": 53800000,
      "unit": "tCO2e",
      "note": "53.80 million tonnes CO2e — Scope 3 GHG emissions in scope of net zero ambition. An additional indirect consumer use category of 51.35 million tCO2e is reported separately but excluded from net zero scope. Total Scope 1, 2 and 3 including indirect consumer use is 105.84 million tCO2e (market-based).",
      "pa

## Day 1 — Observations from Unilever ESRS E1 extraction

**Insight 1: Scope 3 boundary scope choices**
Unilever reports Scope 3 = 53.8M tCO2e in headline figures, but excludes 51.35M of "indirect consumer use" (use phase of sold products) from this number because it sits outside their net zero target boundary. Including it brings total emissions to 105.84M tCO2e. Naive extraction grabs 53.8M and misses the boundary caveat. LLMs can catch this but only if prompted to surface methodology choices alongside numbers.

**Insight 2: Inconsistent Scope 3 category allocation**
Unilever allocates upstream transportation (supplier → Unilever) to Category 1 (purchased goods) instead of Category 4 (upstream transport). This is non-standard. Cross-company comparisons of Scope 3 by category are therefore unreliable without normalization.

**Insight 3: Cost economics**
$0.13 per company-section extraction with Sonnet 4.6 on Bedrock. ~30K input tokens, ~3K output tokens. CSRD applies to ~11,700 EU companies in wave 1. Full ESRS E1 + Taxonomy extraction across all of them = ~$1,500-3,000 in compute. Trivially cheap at scale.

**Insight 4: Output truncation risk**
Hit max_tokens ceiling of 4000 on first call. Long-form structured extractions need either higher token limits or chunked extraction strategies. This is a real production engineering issue.

In [17]:
import json
import re

# Re-run the extraction with a higher max_tokens to avoid truncation
print("Re-running extraction with higher token limit...\n")

response = client.messages.create(
    model="us.anthropic.claude-sonnet-4-6",
    max_tokens=8000,  # doubled from 4000
    messages=[{"role": "user", "content": extraction_prompt}]
)

raw_output = response.content[0].text
print(f"Output length: {len(raw_output):,} chars")
print(f"Output tokens: {response.usage.output_tokens:,}")

# Try to parse as JSON
try:
    # Sometimes Claude wraps in ```json ... ``` despite instructions, strip if so
    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)

    data = json.loads(cleaned)
    print("\n✅ Successfully parsed as JSON")

    # Save to disk
    output_path = f'{PROJECT_ROOT}/data/extracted/unilever_e1.json'
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Saved to {output_path}")

    # Show the data quality flags — most interesting field
    print("\n--- Data quality flags Claude reported ---")
    flags = data.get("data_quality_flags", [])
    if flags:
        for i, flag in enumerate(flags, 1):
            print(f"\n{i}. {flag}")
    else:
        print("(No flags reported)")

except json.JSONDecodeError as e:
    print(f"\n❌ JSON parse failed: {e}")
    print("\nRaw output (last 1000 chars):")
    print(raw_output[-1000:])

Re-running extraction with higher token limit...

Output length: 9,108 chars
Output tokens: 2,843

✅ Successfully parsed as JSON
Saved to /content/drive/MyDrive/csrd-analyzer/data/extracted/unilever_e1.json

--- Data quality flags Claude reported ---

1. Scope 3 total reported as 53.80 million tCO2e (net zero scope, page 248). Summing the disclosed line items — Purchased goods and services (41.79m) + Upstream transport (1.61m) + Downstream leased assets (2.79m) + Use of sold products HFC (1.60m) + End of life (3.70m) + Others (2.31m) — yields 53.80 million tCO2e, which is consistent with the stated total.

2. Scope 3 category mapping is non-standard in places: Unilever explicitly states it classifies supplier-to-Unilever transportation under Category 1 (Purchased goods and services) rather than Category 4 as recommended by the GHG Protocol, making category-level comparisons with other companies unreliable (page 247).

3. The report presents two different 'total Scope 3' figures: 53.80 

## Day 1 — Real findings on Unilever (the memo material)

The `data_quality_flags` field in our extraction caught 7 substantive methodology issues without being asked to look for any of them specifically:

1. **Scope 3 category aggregation hides detail.** "Others" bucket aggregates 7 GHGP categories in a single line, breaking category-level comparability.

2. **Two coexisting "Scope 3 totals" (53.8M vs 105.84M tCO2e).** Boundary exclusions for indirect consumer use create a ~2× difference between headline and actual.

3. **SBTi cross-cutting classifications (E&I, FLAG) don't partition cleanly with GHGP categories.** Naive reconciliation fails.

4. **Fiscal year change (Sep-Sep → Jan-Dec for 2024) breaks YoY comparisons.** Critical for any trend analysis.

5. **Target performance % uses target-scope baseline, not gross baseline.** Reports -72% reduction; gross numbers show -66%. Difference is biogenic fuel + leased vehicle exclusions.

6. **"Shadow carbon price" mentioned without quantification.** Soft-disclosure pattern.

7. **Limited assurance scope ambiguity.** Which specific metrics are KPMG-assured isn't fully clear.

**Implication for memo:** The differentiated LLM use case in CSRD is methodology audit, not value extraction. Numbers are easy to extract; *correctly contextualizing* them requires sophisticated reasoning that LLMs do well — at near-zero cost.

**Cost of this analysis:** $0.26 total (one extraction call + one re-run with higher max_tokens).

In [18]:
# Keywords specific to EU Taxonomy disclosure
taxonomy_signals = [
    "EU Taxonomy",
    "Taxonomy-eligible",
    "Taxonomy-aligned",
    "Taxonomy eligible",
    "Taxonomy aligned",
    "substantial contribution",
    "DNSH",
    "Do No Significant Harm",
    "minimum safeguards",
    "Article 8",
    "Delegated Act",
    "environmental objectives",
    "climate change mitigation",
    "climate change adaptation",
]

# Score each page
taxonomy_scored = []
for p in pages_text:
    text_lower = p["text"].lower()
    score = sum(1 for sig in taxonomy_signals if sig.lower() in text_lower)
    taxonomy_scored.append({"page": p["page"], "score": score, "text": p["text"]})

# Find pages with strong Taxonomy signal
high_signal = [p for p in taxonomy_scored if p["score"] >= 3]
print(f"Found {len(high_signal)} pages with strong Taxonomy signal (score >= 3)")
print(f"\nTop 15 pages by score:")
top = sorted(taxonomy_scored, key=lambda x: x["score"], reverse=True)[:15]
for p in top:
    preview = p["text"][:120].replace("\n", " ")
    print(f"  Page {p['page']} (score {p['score']}): {preview}...")

Found 4 pages with strong Taxonomy signal (score >= 3)

Top 15 pages by score:
  Page 266 (score 7): EU Taxonomy Disclosures OVERVIEW The EU Taxonomy regulation sets out the reporting obligations to  be included in the su...
  Page 268 (score 5): Proportion of operating expenses from products or services associated with Taxonomy-aligned economic  activities – discl...
  Page 267 (score 4): Proportion of capital expenditure from products or services associated with Taxonomy-aligned economic  activities – disc...
  Page 269 (score 4): Proportion of turnover from products or services associated with Taxonomy-aligned economic activities –  disclosure for ...
  Page 296 (score 2): KPMG LLP’s Independent  Assurance Report LIMITED ASSURANCE CONCLUSION We have performed a limited assurance engagement o...
  Page 297 (score 2): For a number of these areas, for example Scope 3 GHG emissions and  Pollution of Air, Water and Soil, there are signific...
  Page 227 (score 1): GENERAL BASIS FOR PREPA

## Day 1 — Sector observation on Taxonomy

Unilever's EU Taxonomy disclosure is 4 pages (pages 266-269) out of a 305-page annual report. Most consumer goods companies will look like this because Taxonomy is heavily weighted toward "transition sectors" — energy, manufacturing, real estate, transport. Selling FMCG products isn't on the eligible activity list to a meaningful degree.

Implication: Headline Taxonomy alignment % is not comparable across sectors. A 2% aligned consumer goods company is not "less green" than a 60% aligned utility — the denominator is fundamentally different. Any product comparing companies on Taxonomy KPIs without sector normalization is misleading by construction.

In [19]:
# Get all pages with at least some Taxonomy signal, in order
taxonomy_pages = [p for p in taxonomy_scored if p["score"] >= 1]

# Combine into one block of text, with page markers
taxonomy_text = "\n\n".join(
    f"[Page {p['page']}]\n{p['text']}" for p in taxonomy_pages
)

estimated_tokens = len(taxonomy_text) // 4
print(f"Taxonomy section: {len(taxonomy_pages)} pages, {len(taxonomy_text):,} chars, ~{estimated_tokens:,} tokens")

# Save to disk
with open(f'{PROJECT_ROOT}/data/text/unilever_taxonomy_section.txt', 'w') as f:
    f.write(taxonomy_text)

print("Saved.")
print(f"\n--- First 1500 chars (likely the overview section) ---")
print(taxonomy_text[:1500])

Taxonomy section: 9 pages, 46,673 chars, ~11,668 tokens
Saved.

--- First 1500 chars (likely the overview section) ---
[Page 227]
GENERAL BASIS FOR PREPARATION
Overview
We have prepared a sustainability statement for Unilever PLC and its 
subsidiary undertakings (Unilever) in accordance with the European 
Sustainability Reporting Standards (the ESRS) as issued by Delegated 
Regulation (EU) 2023/2772 on 31 July 2023. 
The sustainability statement presents information about Unilever’s 
material impacts, risks and opportunities in relation to environmental, 
social and governance matters. The statement comprises four sections: 
■ General Information – summarises our basis of preparation for the 
sustainability statement, including the governance of our 
sustainability strategy and our assessment of our material impacts, 
risks and opportunities (IROs).
■ Environmental Disclosures – provides a consolidated view of our 
processes to identify our material IROs and overarching policies that 


In [20]:
import json, re

with open(f'{PROJECT_ROOT}/data/text/unilever_taxonomy_section.txt', 'r') as f:
    taxonomy_text = f.read()

taxonomy_prompt = f"""You are a sustainability data analyst with deep expertise in EU Taxonomy regulation, extracting structured Taxonomy disclosures from a corporate sustainability report. The report is from Unilever, fiscal year 2024.

Below is the Taxonomy-relevant section of their report (with [Page N] markers).

Your task: extract the following data points into clean JSON. For every numeric value, include the source page number. If a data point is not disclosed, set the value to null. Do NOT guess or infer values.

REQUIRED FIELDS:

1. headline_kpis:
   - turnover_eligible_pct (number with page)
   - turnover_aligned_pct (number with page)
   - capex_eligible_pct (number with page)
   - capex_aligned_pct (number with page)
   - opex_eligible_pct (number with page)
   - opex_aligned_pct (number with page)

2. environmental_objectives_claimed:
   - List of objects, one per objective they claim contribution to. Each object: {{objective_name (one of: climate_mitigation, climate_adaptation, water, circular_economy, pollution, biodiversity), kpi_type (turnover/capex/opex), aligned_pct (number), page}}

3. activities_claimed:
   - List of all specific Taxonomy activities the company claims as eligible or aligned. Each object: {{activity_code (e.g. "7.7"), activity_name (e.g. "Acquisition and ownership of buildings"), kpi_type, eligible_pct, aligned_pct, page}}

4. dnsh_assessment:
   - is_per_activity (true if DNSH is assessed activity-by-activity, false if generic/boilerplate, null if unclear)
   - notable_gaps (list of strings — any DNSH areas where the disclosure is vague or missing)

5. minimum_safeguards:
   - has_dedicated_disclosure (true/false/null)
   - approach_summary (string — short summary of how they demonstrate compliance, max 200 chars)
   - is_substantive (true if backed by specific human rights due diligence, false if only references generic codes of conduct, null if unclear)

6. data_quality_flags:
   Specifically check for these issues, plus any others you notice:
   - INTERNAL CONSISTENCY: Do any KPI percentages exceed 100%? Are aligned percentages ever GREATER than the corresponding eligible percentages? (Aligned must be a subset of eligible — aligned > eligible is a definitional error.)
   - SUM RECONCILIATION: When activities are broken down across environmental objectives, do the per-objective values sum correctly to the headline KPIs? Flag any discrepancies.
   - DISCLOSURE COMPLETENESS: Do they report only eligibility without alignment (or vice versa)? Article 8 requires both.
   - DOUBLE-COUNTING: Is the same activity claimed under multiple environmental objectives in a way that suggests additive reporting errors?
   - BOILERPLATE DNSH: Is DNSH disclosed as a single generic paragraph rather than per-activity, per-objective?
   - SAFEGUARDS DEPTH: Are minimum safeguards demonstrated through specific human rights due diligence processes, or only by reference to generic codes of conduct?
   - SECTOR CONTEXT: Is the disclosure unusually short or thin given the company's sector?

   Each flag should be a specific, page-cited observation. Be concrete. If you find none, return an empty list.

OUTPUT: Return ONLY valid JSON, no markdown, no preamble.

REPORT TEXT:
\"\"\"
{taxonomy_text}
\"\"\"
"""

print(f"Prompt size: ~{len(taxonomy_prompt) // 4:,} tokens")
print("Sending to Claude...\n")

response = client.messages.create(
    model="us.anthropic.claude-sonnet-4-6",
    max_tokens=8000,
    messages=[{"role": "user", "content": taxonomy_prompt}]
)

raw_output = response.content[0].text
print(f"Output: {len(raw_output):,} chars, {response.usage.output_tokens:,} tokens")

cost = (response.usage.input_tokens / 1_000_000 * 3) + (response.usage.output_tokens / 1_000_000 * 15)
print(f"Cost: ${cost:.4f}\n")

# Parse JSON
cleaned = raw_output.strip()
if cleaned.startswith("```"):
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

try:
    data = json.loads(cleaned)
    print("✅ Parsed successfully\n")

    output_path = f'{PROJECT_ROOT}/data/extracted/unilever_taxonomy.json'
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Saved to {output_path}\n")

    # Show headline KPIs
    print("--- Headline KPIs ---")
    print(json.dumps(data.get("headline_kpis", {}), indent=2))

    print("\n--- Data quality flags ---")
    flags = data.get("data_quality_flags", [])
    for i, flag in enumerate(flags, 1):
        print(f"\n{i}. {flag}")

except json.JSONDecodeError as e:
    print(f"❌ JSON parse failed: {e}")
    print(raw_output[-1500:])

Prompt size: ~12,474 tokens
Sending to Claude...

Output: 9,754 chars, 2,968 tokens
Cost: $0.0838

✅ Parsed successfully

Saved to /content/drive/MyDrive/csrd-analyzer/data/extracted/unilever_taxonomy.json

--- Headline KPIs ---
{
  "turnover_eligible_pct": {
    "value": 0,
    "page": 266
  },
  "turnover_aligned_pct": {
    "value": 0,
    "page": 266
  },
  "capex_eligible_pct": {
    "value": 15.1,
    "page": 266
  },
  "capex_aligned_pct": {
    "value": 0,
    "page": 266
  },
  "opex_eligible_pct": {
    "value": 0,
    "page": 266
  },
  "opex_aligned_pct": {
    "value": 0,
    "page": 266
  }
}

--- Data quality flags ---

1. {'issue': 'INTERNAL CONSISTENCY – Per-activity eligible percentages do not sum to headline eligible capex', 'detail': "The per-activity eligible percentages in the table on page 267 sum to approximately 15.1% (0.2+0.1+0.1+0.0+0.1+0.3+0.8+0.2+13.3+0.0+0.0+0.0), which is consistent with the headline 15.1% on page 266. However, the absolute euro amounts s

## Day 1 — Unilever Taxonomy findings (memo material)

**Headline:** €60.8bn FMCG company. Turnover eligible: 0%. Capex eligible: 15.1%. Capex aligned: 0%. Opex: 0%. Effectively no Taxonomy footprint despite scale.

**The 15.1% eligible / 0% aligned gap on capex** is the textbook case for why eligible-vs-aligned matters. ESG databases that report only eligibility produce misleading impressions of corporate environmental performance. The "real" number is zero.

**7 substantive flags surfaced from a single ~12K-token extraction call ($0.08):**

1. Sub-1% capex amounts displayed as "—%" — small line items hidden by rounding presentation, despite having non-zero euro values.
2. 2023 vs 2024 comparison shows methodology change (17.7% → 15.1%) without explanation in narrative.
3. Zero turnover eligibility unexplained for an FMCG company with significant building, packaging, and supply chain assets that could conceivably qualify.
4. Disclosure is thin (~3 pages) given company scale; sector explanation given but no attempt to identify partial eligibility.
5. DNSH assessment is column-of-N/EL with no substantive per-activity, per-objective evaluation. Boilerplate.
6. Minimum safeguards demonstrated by reference to internal policy review, not by specific OECD Guidelines / UNGP mapping. Generic.
7. Double-counting handled correctly (activities 7.1 / 7.2 allocated to CCM only) but circular economy contribution then invisible — methodology trade-off documented in footnote.

**Key insight for memo:**
The flags Claude generated cover (a) presentation issues, (b) consistency issues, (c) completeness issues, and (d) methodology depth issues — all of which are typically caught by senior analysts manually. Generated in 60 seconds for $0.08. The cost asymmetry vs. human review time is dramatic.

In [21]:
import json

# Load both extractions
with open(f'{PROJECT_ROOT}/data/extracted/unilever_e1.json', 'r') as f:
    e1_data = json.load(f)

with open(f'{PROJECT_ROOT}/data/extracted/unilever_taxonomy.json', 'r') as f:
    taxonomy_data = json.load(f)

cross_check_prompt = f"""You are a senior sustainability analyst evaluating the COHERENCE between a company's ESRS E1 climate disclosures and their EU Taxonomy disclosures. The company is Unilever, fiscal year 2024.

You have access to two structured extractions from their report:

=== ESRS E1 EXTRACTION (climate change disclosure) ===
{json.dumps(e1_data, indent=2)}

=== EU TAXONOMY EXTRACTION ===
{json.dumps(taxonomy_data, indent=2)}

Your task: assess whether these two disclosures tell a COHERENT story, or whether there are tensions between them.

Specifically evaluate:

1. TRANSITION PLAN COHERENCE: Does the company's stated climate transition plan (from ESRS E1) align with what their Taxonomy disclosure shows? For example, if they claim a credible decarbonization pathway in E1, does the Taxonomy show meaningful capex allocated to climate-mitigation-aligned activities?

2. TARGET vs. INVESTMENT MISMATCH: The ESRS E1 targets describe future ambition. The Taxonomy capex KPI shows where money is actually being deployed today. Is there a credible link between ambition and current investment, or do they appear disconnected?

3. METHODOLOGY DEPTH ASYMMETRY: Is one disclosure substantially more rigorous than the other? E.g. detailed E1 narrative but boilerplate Taxonomy DNSH? What might that asymmetry suggest?

4. SECTOR FRAMING CONSISTENCY: How does the company explain the gap (if any) between their E1 ambition and their Taxonomy footprint? Is the explanation specific or generic?

5. RED FLAGS: Are there any specific contradictions, suspicious gaps, or signals that one disclosure is treated as a marketing exercise while the other is treated as a compliance exercise?

OUTPUT: Return clean JSON with this structure:

{{
  "overall_coherence": "high" | "medium" | "low",
  "coherence_summary": "2-3 sentence overall assessment",
  "specific_findings": [
    {{
      "category": "transition_plan_coherence" | "target_vs_investment" | "methodology_asymmetry" | "sector_framing" | "red_flag",
      "finding": "specific observation",
      "evidence": "what data points or disclosures support this",
      "severity": "low" | "medium" | "high"
    }}
  ],
  "questions_for_management": [
    "list of 3-5 specific questions an analyst or investor should ask the company about these disclosures"
  ]
}}

Return ONLY the JSON, no preamble.
"""

print(f"Prompt size: ~{len(cross_check_prompt) // 4:,} tokens")
print("Running cross-check analysis...\n")

response = client.messages.create(
    model="us.anthropic.claude-sonnet-4-6",
    max_tokens=4000,
    messages=[{"role": "user", "content": cross_check_prompt}]
)

import re
raw_output = response.content[0].text
cleaned = raw_output.strip()
if cleaned.startswith("```"):
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

cross_data = json.loads(cleaned)

# Save
with open(f'{PROJECT_ROOT}/data/extracted/unilever_crosscheck.json', 'w') as f:
    json.dump(cross_data, f, indent=2)

cost = (response.usage.input_tokens / 1_000_000 * 3) + (response.usage.output_tokens / 1_000_000 * 15)
print(f"Cost: ${cost:.4f}\n")

print(f"=== Overall coherence: {cross_data.get('overall_coherence', 'N/A').upper()} ===")
print(f"\n{cross_data.get('coherence_summary', '')}\n")

print("--- Specific findings ---")
for i, f in enumerate(cross_data.get("specific_findings", []), 1):
    print(f"\n{i}. [{f.get('severity', '?').upper()}] {f.get('category')}")
    print(f"   Finding: {f.get('finding')}")
    print(f"   Evidence: {f.get('evidence')}")

print("\n--- Questions for management ---")
for i, q in enumerate(cross_data.get("questions_for_management", []), 1):
    print(f"{i}. {q}")

Prompt size: ~5,313 tokens
Running cross-check analysis...

Cost: $0.0726

=== Overall coherence: LOW ===

Unilever presents a highly detailed and ambitious ESRS E1 climate narrative — including SBTi-validated targets, a net-zero 2039 commitment, and a multi-lever transition plan — that is structurally disconnected from its EU Taxonomy disclosure, which shows 0% aligned capex, 0% aligned turnover, and 0% aligned opex. The gap between stated decarbonization ambition and Taxonomy-verifiable investment is not adequately explained by the company's sector-framing argument, and the asymmetry in disclosure depth between the two sections suggests they are prepared to different standards of scrutiny. Taken together, the disclosures risk creating a misleading impression of progress: the E1 section reads as a credibility statement while the Taxonomy section reveals an absence of externally verifiable, criteria-tested climate investment.

--- Specific findings ---

1. [HIGH] transition_plan_cohere

## Day 1 — The cross-check is the artifact

Built coherence-checking pipeline that compares ESRS E1 climate disclosure against EU Taxonomy disclosure for the same company. Run on Unilever 2024 report.

**Result:** 9 substantive findings, 5 rated HIGH severity. Cost: $0.07.

**Headline tensions identified:**

1. **Same activities, two stories.** ESRS E1 names solar PV, heat pumps, bioenergy, energy efficiency, vehicle electrification as transition levers. Taxonomy disclosure shows these *same activities* are eligible (CCM 4.1, 4.16, 4.24, 7.3, 6.5) but 0% aligned. The company is investing in the activities but cannot/will not document Taxonomy compliance.

2. **Two contradictory explanations on the same page.** Page 266 explains 0% alignment via "insufficient documentation for substantial contribution" *and* "EU Taxonomy doesn't cover FMCG." These are mutually inconsistent. If structurally ineligible, documentation is irrelevant. If documentation is the issue, the activities are eligible — meaning sector framing is misdirection.

3. **88% of eligible capex is passive building ownership.** Activity CCM 7.7 (Acquisition and ownership of buildings) accounts for 13.3% of 15.1% eligible capex. Active transition activities sum to ~1.8% of total capex. None aligned. Passive property holding is not transition investment.

4. **CNF transition fund (€0.7bn) doesn't appear in Taxonomy.** Climate Transition Action Plan describes €0.4bn added in 2024 to upstream value chain investments. None of this maps to Taxonomy-aligned capex.

5. **Methodology depth asymmetry.** ESRS E1 is KPMG-assured, includes detailed Scope 3 categories, SBTi references, baseline restatements. Taxonomy section is ~3 pages, generic DNSH, generic minimum safeguards. One section was prepared to investor-grade scrutiny; the other wasn't.

**Memo thesis (locked in):**
The differentiated AI use case in CSRD compliance is not data extraction — it is *cross-disclosure coherence audit*. Extraction is increasingly a commodity. Reasoning across two related disclosures to identify methodological tensions, narrative contradictions, and ambition-investment gaps is hard for humans (tedious, requires both frameworks deeply) and easy for well-prompted LLMs. The cost asymmetry is dramatic: ~$0.30 per company per coherence audit vs. hours of senior-analyst time.

**The 5 generated questions for management** are worth attaching to the memo as Appendix A — they're substantively the same questions an audit committee or activist investor would generate, but produced automatically.

In [27]:
import json, re
from pypdf import PdfReader

# ============================================================
# REUSABLE PIPELINE FUNCTIONS
# ============================================================

def extract_pdf_text(pdf_path):
    """Read a PDF and return a list of {page, text} dicts."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append({"page": i + 1, "text": text})
    return pages


def find_relevant_section(pages, signals, min_score=2):
    """Score each page by keyword matches; return concatenated relevant text."""
    scored = []
    for p in pages:
        text_lower = p["text"].lower()
        score = sum(1 for sig in signals if sig.lower() in text_lower)
        scored.append({"page": p["page"], "score": score, "text": p["text"]})
    relevant = [p for p in scored if p["score"] >= min_score]
    section_text = "\n\n".join(f"[Page {p['page']}]\n{p['text']}" for p in relevant)
    return section_text, scored


ESRS_E1_SIGNALS = [
    "ESRS E1", "E1 Climate", "Climate change", "GHG emissions",
    "Scope 1", "Scope 2", "Scope 3", "transition plan", "carbon footprint",
]

TAXONOMY_SIGNALS = [
    "EU Taxonomy", "Taxonomy-eligible", "Taxonomy-aligned",
    "Taxonomy eligible", "Taxonomy aligned", "substantial contribution",
    "DNSH", "Do No Significant Harm", "minimum safeguards",
    "Article 8", "Delegated Act", "environmental objectives",
    "climate change mitigation", "climate change adaptation",
]


def call_claude(prompt, max_tokens=8000):
    """Send a prompt to Claude on Bedrock and return parsed JSON + cost."""
    response = client.messages.create(
        model="us.anthropic.claude-sonnet-4-6",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
    cost = (response.usage.input_tokens / 1_000_000 * 3) + (response.usage.output_tokens / 1_000_000 * 15)
    try:
        data = json.loads(raw)
        return data, cost, response.usage
    except json.JSONDecodeError as e:
        print(f"❌ JSON parse failed: {e}")
        print(f"Last 500 chars:\n{raw[-500:]}")
        return None, cost, response.usage


def build_e1_prompt(company_name, fiscal_year, climate_text):
    return f"""You are a sustainability data analyst extracting structured climate disclosures from a corporate sustainability report. Company: {company_name}, fiscal year {fiscal_year}.

Below is the climate-relevant section (with [Page N] markers).

Extract into clean JSON. For every numeric value, include the source page number. If a data point is not disclosed, set the value to null. Do NOT guess.

REQUIRED FIELDS:

1. ghg_emissions: scope_1_tco2e, scope_2_location_based_tco2e, scope_2_market_based_tco2e, scope_3_total_tco2e (each with value, unit, page), and scope_3_categories (list of {{category_number, category_name, value_tco2e, page}})

2. emissions_targets: has_sbti_validation, sbti_target_description, net_zero_target_year, scope_1_2_reduction_target_pct, scope_1_2_baseline_year, scope_3_reduction_target_pct, scope_3_baseline_year

3. transition_plan: has_transition_plan, key_levers (max 5), capex_aligned_to_transition

4. internal_carbon_price: is_used, price_per_tco2e, currency, scope_of_application

5. methodology: calculation_standard, assurance_provider, assurance_level

6. data_quality_flags: list of specific, page-cited observations about inconsistencies, ambiguities, or notable concerns. Be concrete.

Return ONLY valid JSON, no markdown.

REPORT TEXT:
\"\"\"
{climate_text}
\"\"\"
"""


def build_taxonomy_prompt(company_name, fiscal_year, taxonomy_text):
    return f"""You are a sustainability data analyst with deep expertise in EU Taxonomy regulation. Company: {company_name}, fiscal year {fiscal_year}.

Extract into clean JSON. Page-cite numbers. If not disclosed, null. Do NOT guess.

REQUIRED FIELDS:

1. headline_kpis: turnover_eligible_pct, turnover_aligned_pct, capex_eligible_pct, capex_aligned_pct, opex_eligible_pct, opex_aligned_pct (each with value and page)

2. environmental_objectives_claimed: list of {{objective_name, kpi_type, aligned_pct, page}}

3. activities_claimed: list of {{activity_code, activity_name, kpi_type, eligible_pct, aligned_pct, page}}

4. dnsh_assessment: is_per_activity, notable_gaps

5. minimum_safeguards: has_dedicated_disclosure, approach_summary, is_substantive

6. data_quality_flags:
   Specifically check for:
   - INTERNAL CONSISTENCY: KPIs exceeding 100%; aligned > eligible (definitional error)
   - SUM RECONCILIATION: per-objective values not summing to headline
   - DISCLOSURE COMPLETENESS: only eligibility without alignment, or vice versa
   - DOUBLE-COUNTING: same activity claimed under multiple objectives additively
   - BOILERPLATE DNSH: generic single paragraph rather than per-activity
   - SAFEGUARDS DEPTH: generic policy refs vs specific human rights due diligence
   - SECTOR CONTEXT: disclosure unusually thin/thick for sector
   Each flag: specific, page-cited, concrete.

Return ONLY valid JSON.

REPORT TEXT:
\"\"\"
{taxonomy_text}
\"\"\"
"""


def build_crosscheck_prompt(company_name, fiscal_year, e1_data, taxonomy_data):
    return f"""You are a senior sustainability analyst evaluating COHERENCE between ESRS E1 climate disclosures and EU Taxonomy disclosures. Company: {company_name}, fiscal year {fiscal_year}.

=== ESRS E1 EXTRACTION ===
{json.dumps(e1_data, indent=2)}

=== EU TAXONOMY EXTRACTION ===
{json.dumps(taxonomy_data, indent=2)}

Assess whether these two disclosures tell a COHERENT story. Evaluate:

1. TRANSITION PLAN COHERENCE: Do E1 transition levers correspond to Taxonomy-aligned activities?
2. TARGET vs INVESTMENT: Are E1 targets matched by Taxonomy-aligned capex?
3. METHODOLOGY DEPTH ASYMMETRY: Is one disclosure substantially more rigorous than the other?
4. SECTOR FRAMING CONSISTENCY: Is any sector-explanation specific or generic?
5. RED FLAGS: Specific contradictions or signals of asymmetric preparation.

OUTPUT JSON:
{{
  "overall_coherence": "high" | "medium" | "low",
  "coherence_summary": "2-3 sentence assessment",
  "specific_findings": [
    {{
      "category": "transition_plan_coherence" | "target_vs_investment" | "methodology_asymmetry" | "sector_framing" | "red_flag",
      "finding": "specific observation",
      "evidence": "supporting data points",
      "severity": "low" | "medium" | "high"
    }}
  ],
  "questions_for_management": ["3-5 specific questions"]
}}

Return ONLY the JSON.
"""


def analyze_company(pdf_path, company_name, fiscal_year, output_dir):
    """Full pipeline: PDF → E1 + Taxonomy extractions → coherence cross-check."""
    print(f"\n{'='*60}")
    print(f"Analyzing: {company_name} ({fiscal_year})")
    print(f"{'='*60}\n")

    print("Step 1: Extracting PDF text...")
    pages = extract_pdf_text(pdf_path)
    print(f"  Pages: {len(pages)}")

    print("\nStep 2: Finding ESRS E1 (climate) section...")
    e1_text, e1_scored = find_relevant_section(pages, ESRS_E1_SIGNALS, min_score=2)
    e1_high = [p for p in e1_scored if p["score"] >= 3]
    print(f"  High-signal pages: {len(e1_high)}")
    print(f"  Section size: ~{len(e1_text)//4:,} tokens")

    print("\nStep 3: Finding EU Taxonomy section...")
    tax_text, tax_scored = find_relevant_section(pages, TAXONOMY_SIGNALS, min_score=1)
    tax_high = [p for p in tax_scored if p["score"] >= 3]
    print(f"  High-signal pages: {len(tax_high)}")
    print(f"  Section size: ~{len(tax_text)//4:,} tokens")

    total_cost = 0
    name_slug = company_name.lower().replace(" ", "_")

    print("\nStep 4: Calling Claude for ESRS E1 extraction...")
    e1_prompt = build_e1_prompt(company_name, fiscal_year, e1_text)
    e1_data, cost, _ = call_claude(e1_prompt)
    total_cost += cost
    if e1_data:
        with open(f'{output_dir}/{name_slug}_e1.json', 'w') as f:
            json.dump(e1_data, f, indent=2)
        print(f"  ✅ Saved E1 extraction. Cost: ${cost:.4f}")
        print(f"  Flags found: {len(e1_data.get('data_quality_flags', []))}")

    print("\nStep 5: Calling Claude for Taxonomy extraction...")
    tax_prompt = build_taxonomy_prompt(company_name, fiscal_year, tax_text)
    tax_data, cost, _ = call_claude(tax_prompt)
    total_cost += cost
    if tax_data:
        with open(f'{output_dir}/{name_slug}_taxonomy.json', 'w') as f:
            json.dump(tax_data, f, indent=2)
        print(f"  ✅ Saved Taxonomy extraction. Cost: ${cost:.4f}")
        print(f"  Flags found: {len(tax_data.get('data_quality_flags', []))}")

    if e1_data and tax_data:
        print("\nStep 6: Running coherence cross-check...")
        cross_prompt = build_crosscheck_prompt(company_name, fiscal_year, e1_data, tax_data)
        cross_data, cost, _ = call_claude(cross_prompt, max_tokens=4000)
        total_cost += cost
        if cross_data:
            with open(f'{output_dir}/{name_slug}_crosscheck.json', 'w') as f:
                json.dump(cross_data, f, indent=2)
            print(f"  ✅ Saved coherence analysis. Cost: ${cost:.4f}")
            print(f"  Overall coherence: {cross_data.get('overall_coherence', 'N/A').upper()}")
            print(f"  Findings: {len(cross_data.get('specific_findings', []))}")

    print(f"\n{'='*60}")
    print(f"Total cost for {company_name}: ${total_cost:.4f}")
    print(f"{'='*60}\n")

    return total_cost


print("Pipeline functions defined. Ready to analyze companies.")

Pipeline functions defined. Ready to analyze companies.


In [28]:
analyze_company(
    pdf_path=f'{PROJECT_ROOT}/data/raw/schneider_2024.pdf',
    company_name="Schneider Electric",
    fiscal_year=2024,
    output_dir=f'{PROJECT_ROOT}/data/extracted'
)


Analyzing: Schneider Electric (2024)

Step 1: Extracting PDF text...
  Pages: 676

Step 2: Finding ESRS E1 (climate) section...
  High-signal pages: 36
  Section size: ~82,288 tokens

Step 3: Finding EU Taxonomy section...
  High-signal pages: 22
  Section size: ~55,823 tokens

Step 4: Calling Claude for ESRS E1 extraction...
  ✅ Saved E1 extraction. Cost: $0.3044
  Flags found: 8

Step 5: Calling Claude for Taxonomy extraction...
  ✅ Saved Taxonomy extraction. Cost: $0.2609
  Flags found: 10

Step 6: Running coherence cross-check...
❌ JSON parse failed: Unterminated string starting at: line 69 column 19 (char 13642)
Last 500 chars:
rtunity: the MACC is a substantive analytical tool that could strengthen the DNSH narrative for upstream Scope 3 but is siloed in the E1 climate risk section with no cross-reference to Taxonomy.",
      "evidence": "E1: MACC applied to carbon-intensive raw materials, ~2,540,000 tCO2eq, ~4% of total Scope 3 (p.153-154). Taxonomy: CE DNSH (p.187-188) and CCM

0.654258

In [29]:
# Re-run ONLY the cross-check step for Schneider with higher max_tokens

with open(f'{PROJECT_ROOT}/data/extracted/schneider_electric_e1.json', 'r') as f:
    schneider_e1 = json.load(f)

with open(f'{PROJECT_ROOT}/data/extracted/schneider_electric_taxonomy.json', 'r') as f:
    schneider_tax = json.load(f)

cross_prompt = build_crosscheck_prompt("Schneider Electric", 2024, schneider_e1, schneider_tax)

print("Running coherence cross-check with extended token budget...")
cross_data, cost, usage = call_claude(cross_prompt, max_tokens=8000)

if cross_data:
    with open(f'{PROJECT_ROOT}/data/extracted/schneider_electric_crosscheck.json', 'w') as f:
        json.dump(cross_data, f, indent=2)
    print(f"\n✅ Saved coherence analysis. Cost: ${cost:.4f}")
    print(f"\n=== Overall coherence: {cross_data.get('overall_coherence', 'N/A').upper()} ===")
    print(f"\n{cross_data.get('coherence_summary', '')}\n")

    print(f"--- {len(cross_data.get('specific_findings', []))} specific findings ---\n")
    for i, f in enumerate(cross_data.get("specific_findings", []), 1):
        print(f"{i}. [{f.get('severity', '?').upper()}] {f.get('category')}")
        print(f"   Finding: {f.get('finding')}")
        print(f"   Evidence: {f.get('evidence')[:300]}{'...' if len(f.get('evidence', '')) > 300 else ''}")
        print()

    print("--- Questions for management ---")
    for i, q in enumerate(cross_data.get('questions_for_management', []), 1):
        print(f"{i}. {q[:250]}{'...' if len(q) > 250 else ''}")
        print()
else:
    print("❌ Still failed. Paste me the full error.")

Running coherence cross-check with extended token budget...

✅ Saved coherence analysis. Cost: $0.1095

=== Overall coherence: MEDIUM ===

Schneider Electric's ESRS E1 and EU Taxonomy disclosures are directionally consistent — both anchor the sustainability narrative around electrical infrastructure, energy efficiency, and decarbonisation of the built environment — but a structural coherence gap exists between the ambition of the E1 transition plan and the relatively low 22% Taxonomy-aligned CapEx ratio. The most significant coherence tension is that 90% of revenues are eligible under the Taxonomy yet only 28% are aligned, a gap that is largely explained by the conservative non-alignment of CE 1.2 (35% of revenues), but which is not adequately cross-referenced or reconciled in the E1 narrative. Methodology depth is notably asymmetric: E1 climate data carries reasonable assurance on Scope 1 and 2 and contains granular GHG accounting, while several Taxonomy DNSH assessments rest on quali

## Day 1 — Schneider analysis (the comparison story)

**Verdict: MEDIUM coherence** (vs. Unilever LOW)
- 12 findings (5 HIGH severity)
- Cost: ~$0.66 total for full pipeline including 676-page PDF processing

**Different failure mode categories from Unilever:**

1. **Arithmetic inconsistency:** €41.6M transition capex total ≠ €39.8M + €14.4M itemised breakdown (€12.6M / 30% gap, unexplained, page 135).

2. **Cross-team coordination failure:** Taxonomy section flags 14% of revenue (€5.3B) non-aligned due to RoHS/REACH chemical substance issues; ESRS E1 transition plan makes zero reference to hazardous substance substitution. The two report sections were clearly drafted in isolation.

3. **OpEx denominator manipulation:** 49% aligned OpEx looks impressive, but OpEx defined as only non-capitalized R&D — narrow slice of actual operational expenses. Cross-company OpEx comparisons are structurally invalid.

4. **Cross-framework reconciliation gap:** Same product portfolio quantified with completely different methodologies in E1 (IEA forward-looking 30-year lifetime) vs. Taxonomy (technical screening criteria). No bridge disclosed.

5. **Mid-cycle target revision + methodology change:** LTIP carbon targets revised February 2025 because Scope 3 upstream targets deemed unreachable. Scope 3 Cat 2 +191%, Cat 3 +146% YoY due to methodology changes. Not reflected as restatement in Taxonomy disclosure.

**Comparative insight for memo:**
Unilever and Schneider both got <HIGH coherence ratings, but for completely different reasons. Sector-mismatched reporters fail on transition-plan-vs-alignment gaps; sector-aligned reporters fail on intra-disclosure arithmetic and cross-team coordination. The same pipeline catches both classes of issues. **This is the product story.**

**Cost economics holding up at scale:**
- Unilever (305 pages): ~$0.30
- Schneider (676 pages): ~$0.66
- Linear scaling with report size, all in single-digit dollars per company.

In [30]:
import json

def load_company_data(name_slug, output_dir):
    """Load all three JSON outputs for a company."""
    data = {}
    for stage in ['e1', 'taxonomy', 'crosscheck']:
        path = f'{output_dir}/{name_slug}_{stage}.json'
        try:
            with open(path, 'r') as f:
                data[stage] = json.load(f)
        except FileNotFoundError:
            data[stage] = None
    return data


def fmt_value(field):
    """Pretty-print a {value, page} dict, or just the value."""
    if isinstance(field, dict):
        v = field.get('value')
        p = field.get('page')
        if v is None:
            return "—"
        if p:
            return f"{v} (p.{p})"
        return str(v)
    if field is None:
        return "—"
    return str(field)


def render_company_section(name, data):
    """Render one company's findings as markdown."""
    e1 = data.get('e1') or {}
    tax = data.get('taxonomy') or {}
    cross = data.get('crosscheck') or {}

    md = [f"## {name}\n"]

    # Coherence verdict — the headline
    verdict = cross.get('overall_coherence', 'unknown').upper()
    md.append(f"**Coherence verdict:** {verdict}\n")
    md.append(f"{cross.get('coherence_summary', '')}\n")

    # Headline numbers
    ghg = e1.get('ghg_emissions', {})
    kpis = tax.get('headline_kpis', {})
    md.append("### Headline numbers\n")
    md.append("| Metric | Value |")
    md.append("|---|---|")
    md.append(f"| Scope 1 (tCO2e) | {fmt_value(ghg.get('scope_1_tco2e'))} |")
    md.append(f"| Scope 2 market-based (tCO2e) | {fmt_value(ghg.get('scope_2_market_based_tco2e'))} |")
    md.append(f"| Scope 3 total (tCO2e) | {fmt_value(ghg.get('scope_3_total_tco2e'))} |")
    md.append(f"| Capex Taxonomy-eligible % | {fmt_value(kpis.get('capex_eligible_pct'))} |")
    md.append(f"| Capex Taxonomy-aligned % | {fmt_value(kpis.get('capex_aligned_pct'))} |")
    md.append(f"| Turnover Taxonomy-aligned % | {fmt_value(kpis.get('turnover_aligned_pct'))} |")
    md.append("")

    # Top coherence findings
    findings = cross.get('specific_findings', [])
    high_findings = [f for f in findings if f.get('severity') == 'high']
    md.append(f"### Top coherence findings ({len(high_findings)} HIGH severity, {len(findings)} total)\n")
    for i, f in enumerate(high_findings[:5], 1):
        md.append(f"**{i}. {f.get('category', '').replace('_', ' ').title()}**")
        md.append(f"{f.get('finding', '')}")
        md.append("")

    # Sample data quality flags from each extraction
    e1_flags = e1.get('data_quality_flags', [])
    tax_flags = tax.get('data_quality_flags', [])
    md.append(f"### Disclosure quality flags surfaced\n")
    md.append(f"- ESRS E1: {len(e1_flags)} flags")
    md.append(f"- EU Taxonomy: {len(tax_flags)} flags")
    md.append(f"- Coherence cross-check: {len(findings)} findings")
    md.append("")

    return "\n".join(md)


# Load both companies
unilever_data = load_company_data('unilever', f'{PROJECT_ROOT}/data/extracted')
schneider_data = load_company_data('schneider_electric', f'{PROJECT_ROOT}/data/extracted')

# Build the report
report_md = []
report_md.append("# CSRD Coherence Audit — Comparative Findings\n")
report_md.append("**Pipeline:** ESRS E1 + EU Taxonomy extraction + cross-disclosure coherence audit using Claude Sonnet 4.6 on AWS Bedrock.\n")
report_md.append("**Companies analyzed:** Unilever (FY2024), Schneider Electric (FY2024). Source: published Annual Reports / Universal Registration Documents.\n")
report_md.append("**Approach:** automated PDF section detection → structured data extraction → LLM-based coherence reasoning across two related disclosures.\n")
report_md.append("---\n")

report_md.append(render_company_section("Unilever PLC", unilever_data))
report_md.append("\n---\n")
report_md.append(render_company_section("Schneider Electric", schneider_data))
report_md.append("\n---\n")

# Comparative summary
report_md.append("## Comparative observations\n")
report_md.append("The same pipeline surfaces fundamentally different *categories* of disclosure issues depending on company profile:\n")
report_md.append("- **Sector-mismatched reporters (Unilever, FMCG):** Failure modes concentrate around transition-plan-vs-alignment gaps, contradictory sector-framing explanations, and 0%-aligned activities that map to stated transition levers.")
report_md.append("- **Sector-aligned reporters (Schneider, industrial electrification):** Failure modes concentrate around intra-disclosure arithmetic errors, cross-team coordination gaps, methodology denominator inflation, and missing reconciliation between framework-specific quantifications of the same underlying activities.\n")
report_md.append("Both verdicts (LOW for Unilever, MEDIUM for Schneider) are substantively defensible. Total compute cost: under $1.00 per company.\n")

final = "\n".join(report_md)

# Save it
output_path = f'{PROJECT_ROOT}/outputs/findings_report.md'
with open(output_path, 'w') as f:
    f.write(final)

print(f"✅ Report saved to {output_path}\n")
print("=" * 60)
print(final)
print("=" * 60)

✅ Report saved to /content/drive/MyDrive/csrd-analyzer/outputs/findings_report.md

# CSRD Coherence Audit — Comparative Findings

**Pipeline:** ESRS E1 + EU Taxonomy extraction + cross-disclosure coherence audit using Claude Sonnet 4.6 on AWS Bedrock.

**Companies analyzed:** Unilever (FY2024), Schneider Electric (FY2024). Source: published Annual Reports / Universal Registration Documents.

**Approach:** automated PDF section detection → structured data extraction → LLM-based coherence reasoning across two related disclosures.

---

## Unilever PLC

**Coherence verdict:** LOW

Unilever presents a highly detailed and ambitious ESRS E1 climate narrative — including SBTi-validated targets, a net-zero 2039 commitment, and a multi-lever transition plan — that is structurally disconnected from its EU Taxonomy disclosure, which shows 0% aligned capex, 0% aligned turnover, and 0% aligned opex. The gap between stated decarbonization ambition and Taxonomy-verifiable investment is not adequately

In [33]:
import os
import shutil

STAGING = f'{PROJECT_ROOT}/github_staging'
os.makedirs(STAGING, exist_ok=True)
os.makedirs(f'{STAGING}/prompts', exist_ok=True)
os.makedirs(f'{STAGING}/outputs', exist_ok=True)

# Build README in chunks to avoid triple-quote issues
readme_lines = [
    "# CSRD Coherence Audit",
    "",
    "**An LLM pipeline that audits coherence between ESRS E1 climate disclosures and EU Taxonomy disclosures in published CSRD reports.**",
    "",
    "Most automated CSRD tooling focuses on extracting numbers from disclosures. This project explores a different question: **do the numbers, narratives, and methodology choices in different parts of the same report tell a coherent story?**",
    "",
    "The pipeline ingests a published CSRD-aligned annual report, extracts structured data from both the ESRS E1 (climate change) section and the EU Taxonomy section, then runs an LLM-based coherence audit across the two — surfacing contradictions, methodology asymmetries, and ambition-vs-investment gaps that human reviewers typically catch only after hours of careful reading.",
    "",
    "Built with Claude Sonnet 4.6 on AWS Bedrock.",
    "",
    "---",
    "",
    "## Why this matters",
    "",
    "CSRD compliance is becoming the dominant European sustainability disclosure regime — ~11,700 companies in wave 1 alone. The bottleneck isn't extraction (which is increasingly a commodity capability); it's **whether disclosures hold up under scrutiny**. Auditors, activist investors, and regulators read these reports forensically. Sustainability teams typically don't, because the cost of doing so manually is prohibitive.",
    "",
    "This pipeline brings that forensic audit capability inside the reach of any sustainability team's pre-publication QA process.",
    "",
    "**Two framings, same tool:**",
    "- *External research framing:* point it at published reports → identify disclosure quality issues for analysts, investors, NGOs.",
    "- *Customer-facing framing:* point it at draft disclosures before publication → surface the specific questions auditors and activist investors will ask, with page references, while there's still time to fix or prepare answers.",
    "",
    "The customer-facing framing is the more interesting product opportunity. The cost economics (under $1.00 per company per audit) make it practical to run continuously across a customer's entire CSRD draft cycle.",
    "",
    "---",
    "",
    "## What it does",
    "",
    "For any company's CSRD-aligned report, the pipeline:",
    "",
    "1. **Extracts text** from the PDF and identifies ESRS E1 and EU Taxonomy sections via keyword scoring.",
    "2. **Calls Claude** to extract structured data + self-reported quality flags from each section. Both prompts request page citations on every numeric value and refuse to guess.",
    "3. **Cross-references the two outputs** in a third call, asking Claude to evaluate coherence: do transition levers in E1 correspond to Taxonomy-aligned activities? Do aggressive 2030 targets show up in current Taxonomy-aligned capex? Is one section meaningfully more rigorous than the other?",
    "4. **Returns a verdict** (HIGH / MEDIUM / LOW coherence) plus specific findings, evidence, and a list of follow-up questions an analyst would put to management.",
    "",
    "Total cost per company: **$0.30–$0.66** depending on report size.",
    "",
    "---",
    "",
    "## Findings on two real companies",
    "",
    "Both companies analyzed using the same pipeline, no custom logic:",
    "",
    "### Unilever PLC (FY2024) — Coherence: LOW",
    "- 305-page Annual Report",
    "- 0% Taxonomy-aligned capex despite a detailed, SBTi-validated transition plan",
    "- 9 specific coherence findings (6 HIGH severity)",
    "- Flagged: same activities (solar PV, heat pumps, vehicle electrification) named as transition levers in E1 but 0% aligned in Taxonomy. Two contradictory explanations for non-alignment on the same page. 88% of \"eligible\" capex is passive building ownership.",
    "",
    "### Schneider Electric (FY2024) — Coherence: MEDIUM",
    "- 676-page Universal Registration Document",
    "- 22% Taxonomy-aligned capex, 28% aligned turnover",
    "- 12 specific coherence findings (5 HIGH severity)",
    "- Flagged: arithmetic inconsistency in transition capex breakdown (€39.8M + €14.4M ≠ €41.6M). 14% of revenue non-aligned due to chemical substance compliance issues, completely absent from the E1 transition plan narrative — strongly suggesting the ESRS and Taxonomy teams worked in isolation.",
    "",
    "**The comparative insight:** the two companies fail coherence in *fundamentally different categories*. Sector-mismatched reporters (Unilever, FMCG) fail on transition-plan-vs-alignment gaps and contradictory sector-framing. Sector-aligned reporters (Schneider, industrial) fail on intra-disclosure arithmetic and cross-team coordination. Different risk profiles call for different audit-readiness checks. The same pipeline catches both.",
    "",
    "See `outputs/findings_report.md` for the full comparative report.",
    "",
    "---",
    "",
    "## How it's built",
    "",
    "```",
    "PDF → text extraction (pypdf) →",
    "keyword-based section detection →",
    "Claude Sonnet 4.6 extraction (ESRS E1) →",
    "Claude Sonnet 4.6 extraction (EU Taxonomy) →",
    "Claude Sonnet 4.6 coherence cross-check →",
    "structured JSON outputs + markdown report",
    "```",
    "",
    "Deliberately simple. No vector DB, no RAG, no agent framework. The point was to understand the data problem before reaching for fancy infrastructure.",
    "",
    "The three prompts are the engineering work. They live in `prompts/` as standalone text files for inspection.",
    "",
    "---",
    "",
    "## Repo contents",
    "",
    "- `notebook.ipynb` — the full pipeline as a Colab notebook",
    "- `prompts/` — the three Claude prompts as standalone text files",
    "- `outputs/` — JSON extractions and findings report from the two companies analyzed",
    "- `requirements.txt` — Python dependencies",
    "",
    "---",
    "",
    "## Limitations and what I'd build next",
    "",
    "This is a weekend prototype, not a production tool. Real limitations:",
    "",
    "1. **Keyword-based section detection is brittle.** Different report formats (Universal Registration Document vs. Annual Report vs. standalone Sustainability Report) structure these sections differently. A production version would use document layout analysis or an LLM-based section classifier.",
    "2. **Two extractions, one cross-check.** Real CSRD coherence audit would extend across all 12 ESRS standards plus Taxonomy plus the financial statements. Same pipeline shape, more pairings.",
    "3. **No evaluation framework.** I have anecdotal evidence the cross-check produces useful findings on two companies. Production deployment requires a benchmark dataset of disclosures with known issues, and systematic measurement of recall on those issues.",
    "4. **No audit trail logging.** Production sustainability tooling needs full reproducibility — model version, prompt version, source document hash, every reasoning step. Trivial to add.",
    "",
    "If I were continuing this in a customer-facing role, the highest-leverage extensions would be (a) building the eval benchmark, (b) extending coverage to all ESRS standards, and (c) integrating with sustainability teams' pre-publication review workflows so findings can be triaged and routed to the right disclosure owner.",
    "",
    "---",
    "",
    "## About this project",
    "",
    "Built as a portfolio piece by Ketki Sawant ([LinkedIn](https://www.linkedin.com/in/ketki-sawant-335aba1b5/)). I spent two years at Morningstar Sustainalytics on EU Taxonomy and reporting frameworks, and I'm now pursuing a Master of Quantitative Economics at UCLA. This project sits at the intersection of those two things: domain depth on the disclosure regimes, plus AI tooling for the parts of the workflow that don't scale with human effort.",
    "",
]

with open(f'{STAGING}/README.md', 'w') as f:
    f.write("\n".join(readme_lines))
print("✅ README.md created")

# requirements.txt
with open(f'{STAGING}/requirements.txt', 'w') as f:
    f.write("boto3>=1.34\nanthropic>=0.40\npypdf>=4.0\n")
print("✅ requirements.txt created")

# Prompt files — built as plain string lists too
e1_lines = [
    "ESRS E1 Climate Disclosure Extraction Prompt",
    "=" * 50,
    "Substituted at runtime: company_name, fiscal_year, climate_text",
    "",
    "You are a sustainability data analyst extracting structured climate disclosures from a corporate sustainability report. Company: <company_name>, fiscal year <fiscal_year>.",
    "",
    "Below is the climate-relevant section (with [Page N] markers).",
    "",
    "Extract into clean JSON. For every numeric value, include the source page number. If a data point is not disclosed, set the value to null. Do NOT guess.",
    "",
    "REQUIRED FIELDS:",
    "",
    "1. ghg_emissions: scope_1_tco2e, scope_2_location_based_tco2e, scope_2_market_based_tco2e, scope_3_total_tco2e (each with value, unit, page), and scope_3_categories (list of objects with category_number, category_name, value_tco2e, page)",
    "",
    "2. emissions_targets: has_sbti_validation, sbti_target_description, net_zero_target_year, scope_1_2_reduction_target_pct, scope_1_2_baseline_year, scope_3_reduction_target_pct, scope_3_baseline_year",
    "",
    "3. transition_plan: has_transition_plan, key_levers (max 5), capex_aligned_to_transition",
    "",
    "4. internal_carbon_price: is_used, price_per_tco2e, currency, scope_of_application",
    "",
    "5. methodology: calculation_standard, assurance_provider, assurance_level",
    "",
    "6. data_quality_flags: list of specific, page-cited observations about inconsistencies, ambiguities, or notable concerns. Be concrete.",
    "",
    "Return ONLY valid JSON, no markdown.",
]
with open(f'{STAGING}/prompts/01_esrs_e1_extraction.txt', 'w') as f:
    f.write("\n".join(e1_lines))

tax_lines = [
    "EU Taxonomy Disclosure Extraction Prompt",
    "=" * 50,
    "Substituted at runtime: company_name, fiscal_year, taxonomy_text",
    "",
    "You are a sustainability data analyst with deep expertise in EU Taxonomy regulation. Company: <company_name>, fiscal year <fiscal_year>.",
    "",
    "Extract into clean JSON. Page-cite numbers. If not disclosed, null. Do NOT guess.",
    "",
    "REQUIRED FIELDS:",
    "",
    "1. headline_kpis: turnover_eligible_pct, turnover_aligned_pct, capex_eligible_pct, capex_aligned_pct, opex_eligible_pct, opex_aligned_pct (each with value and page)",
    "",
    "2. environmental_objectives_claimed: list of objects with objective_name, kpi_type, aligned_pct, page",
    "",
    "3. activities_claimed: list of objects with activity_code, activity_name, kpi_type, eligible_pct, aligned_pct, page",
    "",
    "4. dnsh_assessment: is_per_activity, notable_gaps",
    "",
    "5. minimum_safeguards: has_dedicated_disclosure, approach_summary, is_substantive",
    "",
    "6. data_quality_flags:",
    "   Specifically check for:",
    "   - INTERNAL CONSISTENCY: KPIs exceeding 100%; aligned > eligible (definitional error)",
    "   - SUM RECONCILIATION: per-objective values not summing to headline",
    "   - DISCLOSURE COMPLETENESS: only eligibility without alignment, or vice versa",
    "   - DOUBLE-COUNTING: same activity claimed under multiple objectives additively",
    "   - BOILERPLATE DNSH: generic single paragraph rather than per-activity",
    "   - SAFEGUARDS DEPTH: generic policy refs vs specific human rights due diligence",
    "   - SECTOR CONTEXT: disclosure unusually thin/thick for sector",
    "   Each flag: specific, page-cited, concrete.",
    "",
    "Return ONLY valid JSON.",
]
with open(f'{STAGING}/prompts/02_eu_taxonomy_extraction.txt', 'w') as f:
    f.write("\n".join(tax_lines))

cross_lines = [
    "Coherence Cross-Check Prompt",
    "=" * 50,
    "Substituted at runtime: company_name, fiscal_year, e1_data (JSON), taxonomy_data (JSON)",
    "",
    "You are a senior sustainability analyst evaluating COHERENCE between ESRS E1 climate disclosures and EU Taxonomy disclosures. Company: <company_name>, fiscal year <fiscal_year>.",
    "",
    "=== ESRS E1 EXTRACTION ===",
    "<e1_data>",
    "",
    "=== EU TAXONOMY EXTRACTION ===",
    "<taxonomy_data>",
    "",
    "Assess whether these two disclosures tell a COHERENT story. Evaluate:",
    "",
    "1. TRANSITION PLAN COHERENCE: Do E1 transition levers correspond to Taxonomy-aligned activities?",
    "2. TARGET vs INVESTMENT: Are E1 targets matched by Taxonomy-aligned capex?",
    "3. METHODOLOGY DEPTH ASYMMETRY: Is one disclosure substantially more rigorous than the other?",
    "4. SECTOR FRAMING CONSISTENCY: Is any sector-explanation specific or generic?",
    "5. RED FLAGS: Specific contradictions or signals of asymmetric preparation.",
    "",
    "Output JSON with: overall_coherence (high/medium/low), coherence_summary (string), specific_findings (list of objects with category, finding, evidence, severity), questions_for_management (list of strings).",
    "",
    "Return ONLY the JSON.",
]
with open(f'{STAGING}/prompts/03_coherence_crosscheck.txt', 'w') as f:
    f.write("\n".join(cross_lines))

print("✅ 3 prompt files created")

# Copy outputs
extracted = f'{PROJECT_ROOT}/data/extracted'
for f in os.listdir(extracted):
    if f.endswith('.json'):
        shutil.copy(f'{extracted}/{f}', f'{STAGING}/outputs/{f}')

shutil.copy(f'{PROJECT_ROOT}/outputs/findings_report.md', f'{STAGING}/outputs/findings_report.md')
print("✅ Outputs copied")

# Summary
print("\n" + "=" * 60)
print(f"GitHub staging folder ready at:\n  {STAGING}")
print("=" * 60)
print("\nContents:")
for root, dirs, files in os.walk(STAGING):
    level = root.replace(STAGING, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in files:
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f'{subindent}{f} ({size_kb:.1f} KB)')

✅ README.md created
✅ requirements.txt created
✅ 3 prompt files created
✅ Outputs copied

GitHub staging folder ready at:
  /content/drive/MyDrive/csrd-analyzer/github_staging

Contents:
github_staging/
  README.md (7.0 KB)
  requirements.txt (0.0 KB)
  prompts/
    01_esrs_e1_extraction.txt (1.4 KB)
    02_eu_taxonomy_extraction.txt (1.5 KB)
    03_coherence_crosscheck.txt (1.1 KB)
  outputs/
    unilever_e1.json (8.9 KB)
    unilever_taxonomy.json (9.6 KB)
    unilever_crosscheck.json (13.8 KB)
    schneider_electric_e1.json (10.4 KB)
    schneider_electric_taxonomy.json (19.3 KB)
    schneider_electric_crosscheck.json (19.3 KB)
    findings_report.md (11.0 KB)
